# Agentic AI Bootcamp
---

## Learning Objectives

1. Understand the capabilities of NVIDIA Inference Microservices (NIMs) and how to use NIM endpoints for accelerated AI inference.
2. Understand the Model Context Protocol (MCP) and how it enables standardized communication tools and data sources.
3. Create MCP servers using the low-level Server API with tool definitions and input schemas.
4. Build stateful chatbot workflows using LangGraph's StateGraph with NVIDIA NIM as the LLM backend, and implement structured output for parseable responses.
5. Use the NVIDIA Nemo Agent Toolkit to connect MCP servers with NIM endpoints and build intent-routed agent workflows.
6. Apply learned concepts in a challenge exercise to build a complete agentic application.
7. Create an agent in OpenClaw that utilises the Nemo Agent Toolkit workflow completed in the challenge.

## Table of Contents

1. [Using NVIDIA Inference Microservices (NIMs)](jupyter_notebook/01_inference_endpoint.ipynb)
2. [Introduction to Model Context Protocol (MCP)](jupyter_notebook/02_introduction_mcp.ipynb)
3. [Low Level MCP Server](jupyter_notebook/03_low_level_mcp.ipynb)
4. [Introduction to LangGraph](jupyter_notebook/04_langraph_agent.ipynb)
5. [NeMo Agent Toolkit (NAT)](jupyter_notebook/05_nemo_agent_toolkit.ipynb)
6. [Challenge Overview](jupyter_notebook/06_challenge.ipynb)

## The digital concierge

#### Prologue: A Store with Too Many Questions

Harmony Records had been selling music for twenty years — vinyl, CDs, and eventually digital downloads. When the company finally launched an online store, the emails started flooding in. Customers wanted to know which albums were in stock, why their invoices looked wrong, whether they could get refunds on purchases they regretted. The support team was overwhelmed.

The store manager, Elena, had heard about AI agents. Not the simple chatbots that parroted FAQ answers, but genuine reasoning systems that could look up data, take actions, and hand off to humans when genuinely uncertain. She wanted one. She called in a solutions architect named Marcus.

"We'll call her Aria," Elena said. "She'll handle everything the support team spends half their day on."

Marcus smiled. "She'll be more than a chatbot. Let's build her right."

---

#### Chapter One: Finding a Voice

The first thing Aria needed was a voice — a language model sharp enough to understand what customers actually meant, not just what they literally typed.

Marcus started by connecting to NVIDIA's inference endpoints: containerized, pre-optimized language models that could be accessed through a standard API. He tested responses. He tuned how the model was invoked. He learned how tokens flowed in and answers flowed out, and he learned the difference between a model running in the cloud versus one deployed on local hardware under the store's full control.

Aria's voice took shape. She could read a question and produce a coherent reply. But she had no memory, no access to the store's data, and no ability to act. She was eloquent and useless.

"A voice without hands," Marcus said. "Let's give her some tools."

---

#### Chapter Two: Learning to Use Tools

The store's database held everything: customers, invoices, albums, artists, tracks, and purchase histories. But the model couldn't query it directly. Marcus needed a way to expose these capabilities to Aria as discrete, callable actions — things she could invoke on demand.

He discovered the **Model Context Protocol**, an open standard designed exactly for this purpose. MCP worked like a contract: a server defined what tools were available and what each one expected as input. Aria's reasoning layer, acting as a client, could discover those tools and invoke them during a conversation.

Marcus built a first MCP server quickly, using high-level decorators that turned ordinary Python functions into tools with almost no extra ceremony. He tested it in a simple chat loop. Aria could now search for an artist by name. She could retrieve a track listing. She could count how many albums a particular label had released.

But as he imagined Aria handling real customer requests — many at once, over a reliable network — Marcus knew he needed more control. He rebuilt the invoice-handling server at a lower level: explicit input schemas, careful validation, deliberate error handling, and an HTTP transport layer that could be reached from anywhere on the network. The high-level approach had been elegant for prototyping. The low-level approach was what production demanded.

Aria now had hands. She still didn't know how to plan.

---

#### Chapter Three: Learning to Think in Steps

A customer writes: *"I bought the wrong version of an album last week, and I want a refund, but can you also tell me what the original pressing year was before I cancel?"*

That single message contains at least three things: a question about product history, a conditional, and a refund request. A single tool call couldn't handle it. Aria needed to reason in steps — retrieve information, hold it in memory, decide what to do next, and loop until the task was complete.

Marcus introduced **LangGraph**, a framework for building stateful reasoning graphs. He modeled Aria's thinking as a directed graph: nodes were actions (call the LLM, invoke a tool, route to a specialist), and edges were the logic connecting them. The state — everything Aria knew about the current conversation — flowed through this graph and was updated at each step.

Now Aria could handle multi-turn conversations. A thread of messages accumulated context. If she called a tool and got back partial information, she could continue reasoning rather than stopping cold. If the conversation paused — because a human supervisor needed to approve a large refund — she could wait, and resume exactly where she left off.

Marcus added a crucial feature: **agent skills**. Rather than loading Aria's full knowledge of the music database into memory at startup, she was given a directory of skill descriptions. She read only what she needed, when she needed it. When a customer asked about track metadata, she fetched the relevant skill instructions, used them to formulate the right database query, and answered cleanly.

Aria could now plan, act, remember, and pause. But Marcus had been writing a lot of glue code — Python files that wired model calls to tool invocations, routing logic hand-coded for every possible path. It was fragile and hard to extend.

---

#### Chapter Four: Declaring Her Personality

Marcus had a conversation with Elena about the future. Not just the refund questions and catalog lookups that were already working — but what Aria would become. More agents. More tools. A new agent for promotional recommendations. Another for tracking shipments.

"Every time we add one, you'll have to rewrite half the code," Marcus said. "There's a better way."

He discovered the **NeMo Agent Toolkit**, a framework that let him describe an agent's entire configuration in a YAML file rather than in hand-wired Python. The model, the tools, the routing logic, the MCP servers it should connect to — all declared. When Marcus wanted to adjust Aria's behavior, he edited a configuration file, not a codebase.

He wired up an intent classifier — a small, fast model that read each incoming message and decided which type of request it was before routing it to the appropriate agent. General music questions went one way. Invoice and refund requests went another. Anything that fell outside those categories was flagged for a human.

He also connected **Phoenix**, an observability platform that recorded every decision Aria made: which tool she called, what the LLM said at each step, how long each stage took. When something went wrong, Marcus could replay the trace and see exactly where the reasoning had broken down.

Aria was now observable, configurable, and extensible.

---

#### Chapter Five: The First Real Day

The morning Aria went live, the support team watched nervously. The first customer message arrived at 9:04 AM.

*"Hi, I'm looking for all albums by Miles Davis that are available in your catalog."*

The intent classifier read the message: general catalog question. Routed to the QNA agent. Aria invoked her catalog search skill, queried the database, retrieved twelve albums, and replied with a clean list in under two seconds.

At 9:11 AM: *"I have an invoice from last month for an order I never received. I want a refund."*

Intent classifier: refund request. Routed to the refund agent. Aria called the invoice search tool, located the order, confirmed it had never been fulfilled, and initiated the refund workflow. She surfaced a summary to the human supervisor for approval before finalizing — just as Marcus had designed.

At 9:47 AM: *"What do you think about the state of jazz today?"*

Intent classifier: out of scope. Aria responded politely that she was specialized for catalog and order questions, and offered to connect the customer with a human if they needed help with something else.

By noon, the support team had handled fewer than a dozen escalations, down from the usual fifty.

Elena stopped by Marcus's desk. "She's doing it."

"She's doing the easy parts," Marcus said. "The interesting work is what comes next — when the questions get harder, when the edge cases stack up, when customers start asking things we didn't anticipate."

He pulled up the Phoenix trace view. Aria's decision graph glowed on the screen — nodes lighting up as each message moved through classification, routing, tool invocation, and response.

"But now we can see exactly what she's doing," he said. "And that means we can teach her."

---

#### Epilogue: What Aria Taught Them

Building Aria had taken Marcus and Elena through a complete arc of ideas.

They had learned that a language model, however capable, is incomplete without tools — and that the way you expose those tools matters enormously. Decorators are fast. Low-level schemas are reliable. Production systems eventually need both.

They had learned that agents aren't scripts. They're stateful reasoning processes that need to hold context, route decisions, pause for human input, and resume cleanly. A graph that you can see and trace is worth more than a loop buried in code.

They had learned that hand-wired logic doesn't scale. Declarative configuration — describing what an agent does rather than programming every step of how — made the difference between a prototype and a maintainable system.

And they had learned that observability isn't optional. An agent you can't trace is an agent you can't improve.

Aria wasn't perfect. She sometimes misclassified an unusual request, occasionally needed a human to step in, and had opinions about the edge cases in the invoice schema that Marcus still found mildly frustrating. But she was real, she was running, and she was getting better.

The support team went home that first Friday without a pile of unanswered tickets waiting for Monday.


--- 
## Licensing 

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.